# Optimization & Convexity

## What's covered

- **Stationary points** and the first-order necessary condition `∇f = 0`
- **Second-order conditions** — the Hessian test for min, max, saddle
- **Convexity** — why convex problems are "easy" and how to recognize them
- **Gradient descent** — algorithm, learning rate, convergence
- **Momentum** and **Adam** — practical accelerations
- **Newton's method** revisited — when to spend the second-order budget
- **Saddle points in high dimensions** — the real problem of deep learning
- Where this appears in ML — every training run


## Stationary points

A **stationary point** of a differentiable function `f: R^n → R` is any `x*` where

$$
\nabla f(\mathbf{x}^*) = \mathbf{0}
$$

In words: every partial derivative is zero. Locally, the function is flat.

**First-order necessary condition.** If `x*` is a local minimum (or maximum) of `f`, then `x*` is a stationary point. Equivalently: **gradient descent's only way to know it has converged is `∇f = 0`** (or within tolerance).

The converse is *not* true. Stationary points include minima, maxima, **saddle points**, and weird degenerate cases. Just because the gradient is zero doesn't mean you're at a minimum. The whole drama of optimization in high dimensions — especially deep learning — is sorting which kind of stationary point you've found.


## Second-order conditions

To distinguish minima from maxima and saddles, look at the Hessian (notebook 5).

**Sufficient condition for a local minimum.** If `∇f(x*) = 0` *and* `H_f(x*)` is **positive definite** (all eigenvalues strictly positive), then `x*` is a strict local minimum.

**Sufficient condition for a local maximum.** Same as above but `H_f(x*)` is **negative definite**.

**Saddle point.** If `H_f(x*)` has **both** positive and negative eigenvalues, `x*` is a saddle — a minimum along some directions, a maximum along others. The function looks like a Pringles chip.

**Degenerate cases.** If `H_f(x*)` is only positive *semi*definite (zero eigenvalues), the second-order test is inconclusive; you need higher-order information.

The Hessian-eigenvalue picture is **the** local geometry of the loss landscape:

- All eigenvalues large and positive → **sharp** minimum (steep bowl). Tends to generalize worse.
- All eigenvalues small and positive → **flat** minimum (wide bowl). Tends to generalize better; this is the basis of methods like SAM (sharpness-aware minimization).
- Mixed signs → saddle. The drag of high-dimensional optimization.


## Convexity

A function `f: R^n → R` is **convex** if for every pair of points and every `t ∈ [0, 1]`:

$$
f(t \mathbf{x} + (1 - t) \mathbf{y}) \le t f(\mathbf{x}) + (1 - t) f(\mathbf{y})
$$

Geometrically: the line segment between two points on the graph lies on or above the graph. The function is **strictly convex** if the inequality is strict whenever `x ≠ y`.

**Two pleasant consequences of convexity.**

1. **Every local minimum is a global minimum.** No bad local minima. You will never get stuck in the wrong valley.
2. **Stationary points are exactly the minima.** `∇f = 0` is both necessary and sufficient.

**Tests for convexity.** Two practical ones:

- **First-order test** — `f` is convex iff `f(y) ≥ f(x) + ∇f(x)^T (y - x)` for every `x, y`. In words: the tangent plane sits everywhere on or below the graph.
- **Second-order test** — `f` is convex iff `H_f(x)` is **positive semidefinite for every `x`**. This is the most-used test in practice — you check that the Hessian is PSD everywhere.

**ML examples of convex functions:**

- **Linear regression / MSE.** `L(w) = (1/n) ||y - Xw||^2`. Hessian `(2/n) X^T X` is always PSD. Convex.
- **Logistic regression** (no regularization). `L(w) = Σ log(1 + exp(-y_i x_i · w))`. Convex.
- **SVM hinge loss.** Convex (and piecewise linear).
- **Lasso, ridge.** Convex (sum of convex functions stays convex).

**ML examples of non-convex functions:**

- **Neural networks.** Loss is non-convex in the weights — many local minima, many saddles. Yet they train. *Why neural networks train despite non-convexity is one of the most-studied questions in ML theory.*
- **k-Means.** Loss is non-convex in the cluster centers. Local minima are real.
- **Mixture models.** Non-convex log-likelihood.

The story of modern ML is "non-convex problems are not the catastrophe theory once feared." Practical gradient methods find good (if not globally optimal) solutions in spite of non-convexity.


In [ ]:
import numpy as np

# Quick visual check of convexity via the second-order test
# f(x) = x^4 - x^2: NOT convex (has two minima and a local max at 0)
# f(x) = x^2 + log(1 + exp(-x)): convex (sum of convex)

def f1(x): return x ** 4 - x ** 2          # non-convex
def f1_pp(x): return 12 * x ** 2 - 2

def f2(x): return x ** 2 + np.log1p(np.exp(-x))   # convex
def f2_pp(x):
    s = 1 / (1 + np.exp(-x))                       # sigmoid
    return 2 + s * (1 - s)                          # 2 + sigmoid' = strictly positive

print("f1(x) = x^4 - x^2 — second derivative:")
for x in [-1, -0.5, 0, 0.5, 1]:
    print(f"  x = {x}: f''(x) = {f1_pp(x):.2f}  (negative at 0 — function not convex)")

print("\nf2(x) = x^2 + log(1 + e^-x) — second derivative:")
for x in [-3, -1, 0, 1, 3]:
    print(f"  x = {x}: f''(x) = {f2_pp(x):.4f}  (always positive — convex)")


## Gradient descent

The vanilla algorithm:

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \, \nabla L(\mathbf{w}_t)
$$

`α > 0` is the **learning rate** (also called the **step size**). The intuition is direct from notebook 3: `-∇L` is the direction of steepest descent, `α` controls how far to step.

**Convergence guarantees (informal).**

- **Convex `L` with Lipschitz gradient** (`||∇L(x) - ∇L(y)|| ≤ M ||x - y||`): convergence with rate `O(1/t)` to the global minimum if `α ≤ 1/M`. Faster with momentum.
- **Strongly convex `L`** (Hessian eigenvalues bounded below by some `μ > 0`): linear convergence; error shrinks by a constant factor each iteration.
- **Non-convex `L`**: convergence to a stationary point under mild conditions. No promises about which one.

**Learning-rate effects.** Three regimes:

- **Too small** — convergence crawls. Many iterations to reach the minimum.
- **Just right** — fastest convergence. For quadratic losses, optimal `α ≈ 2 / (λ_min + λ_max)` where the `λ`'s are Hessian eigenvalues.
- **Too large** — overshoots, diverges, or oscillates. The first-order Taylor approximation breaks down outside the trust radius.

**Stochastic gradient descent (SGD).** Replace the full gradient `∇L` with a mini-batch estimate. Cheaper per step, noisier per step. The noise turns out to *help* — it lets you escape saddle points and bad local minima that exact gradient descent might get trapped near.


In [ ]:
# Gradient descent on a convex quadratic: f(x, y) = x^2 + 2 y^2
# Optimal solution: (0, 0). Hessian eigenvalues: 2 and 4. Best alpha ≈ 2/(2+4) = 1/3.

def grad_f(xy): return np.array([2 * xy[0], 4 * xy[1]])

for alpha, label in [(0.05, "too small"), (0.33, "just right"), (0.6, "too large")]:
    xy = np.array([5.0, 5.0])
    history = [xy.copy()]
    for _ in range(15):
        xy = xy - alpha * grad_f(xy)
        history.append(xy.copy())
    norms = [np.linalg.norm(h) for h in history]
    print(f"α = {alpha:<4} ({label:<10}):  ||xy|| over time = {[f'{n:.3f}' for n in norms[:8]]}{'...' if len(norms)>8 else ''}")


## Momentum and Adam

Plain SGD takes a small step downhill at every iteration. **Momentum** lets the optimizer build up velocity in consistent directions, like a ball rolling down a valley:

$$
\mathbf{v}_{t+1} = \beta \mathbf{v}_t + \nabla L(\mathbf{w}_t), \qquad \mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \mathbf{v}_{t+1}
$$

`β ∈ [0, 1)` is the **momentum coefficient** (usually `0.9`). Past gradients accumulate (decayed by `β`) into the velocity, which then drives the step. The result: accelerated motion in directions where the gradient is consistent, dampened motion in directions where it oscillates. **Polyak heavy-ball** is the original form; **Nesterov momentum** is a slightly smarter variant.

**Adam.** Combines momentum with **per-parameter learning rates** scaled by recent gradient *magnitude*. Roughly:

$$
\mathbf{m}_t = \beta_1 \mathbf{m}_{t-1} + (1 - \beta_1) \nabla L \quad \text{(first-moment estimate)}
$$
$$
\mathbf{v}_t = \beta_2 \mathbf{v}_{t-1} + (1 - \beta_2) \nabla L \odot \nabla L \quad \text{(second-moment estimate)}
$$
$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \alpha \, \hat{\mathbf{m}}_t / (\sqrt{\hat{\mathbf{v}}_t} + \epsilon)
$$

`\hat m` and `\hat v` are bias-corrected versions; `ε` prevents division by zero. The key intuition: **parameters with consistently large gradients get smaller effective learning rates**, parameters with small gradients get larger ones. Adam adapts automatically to per-parameter geometry.

Adam is the default optimizer for most modern deep learning. SGD with momentum is still preferred in computer vision (CNN training) because it tends to find flatter minima.

Both momentum and Adam are still "first-order" — they only use gradient information, but they accumulate gradients in ways that mimic second-order methods cheaply.


## Newton's method, revisited

Notebook 6 derived Newton's update from the second-order Taylor expansion:

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - H_L(\mathbf{w}_t)^{-1} \nabla L(\mathbf{w}_t)
$$

**Pros.**

- **Quadratic convergence** near a minimum — much faster than first-order methods.
- **Scale-invariant** — Newton's step ignores parameter rescalings.
- For a quadratic objective, Newton converges in **one step** from any starting point.

**Cons (in ML).**

- **Memory** — storing `H` is `O(n^2)`. With `n = 10^9` parameters, that's `10^18` entries. Infeasible.
- **Compute** — inverting `H` is `O(n^3)`.
- **Not safe far from a minimum** — if `H` isn't positive definite, Newton's step may point uphill or be enormous.

**ML compromises:**

- **L-BFGS** — limited-memory BFGS — stores only the last `m` rank-1 updates to approximate `H^{-1}`. `O(mn)` memory. Standard in classical optimization libraries.
- **Natural gradient** — replace `H` with the Fisher information matrix, which is PSD by construction and has nice properties for probability distributions.
- **K-FAC, Shampoo** — block-diagonal approximations to `H` based on layer structure.
- **Hessian-free / conjugate gradient** — compute `H v` for vectors `v` via Pearlmutter's trick without forming `H`.
- **Diagonal Hessian methods** — keep only `diag(H)`. Adam approximates this with `E[(∇L)^2]`.

For most deep learning, first-order Adam is "good enough." Second-order methods shine for smaller models, fine-tuning, and convex sub-problems inside larger pipelines.


## Saddle points — the real challenge in high dimensions

Classical optimization worried about local minima. In high dimensions, **saddles are the real obstacle.**

A saddle point has a Hessian with both positive and negative eigenvalues. With `n` parameters, picking a *random* stationary point, the probability that *all* `n` eigenvalues happen to be positive becomes astronomically small. So in deep networks, **the overwhelming majority of stationary points are saddles.**

**What this means for training:**

- Pure gradient descent stalls near saddles — the gradient is small but the saddle isn't a minimum.
- SGD's *noise* helps escape, by pushing parameters into directions of negative curvature.
- Momentum can power through saddle regions because it doesn't stop just because the gradient briefly vanishes.
- Second-order methods can detect saddles (mixed eigenvalues) and escape deliberately.

**The optimistic view (now standard in deep-learning theory).** In overparameterized networks, the loss landscape — though non-convex — has a "nice" structure: most local minima found by SGD are roughly equivalent (similar loss), and the saddles, while abundant, can be escaped by noise or momentum. This is why deep networks train despite their non-convexity. The "loss landscape" papers of the late 2010s explored this in detail.

**The skeptical view.** Sharpness vs flatness of minima matters; SGD's noise has implicit regularization effects that gradient descent doesn't; learning rate schedules and warmup interact with curvature in ways still being understood. None of this should reduce your respect for empirical practice — but the math is real and rich.


In [ ]:
# Saddle point demo: f(x, y) = x^2 - y^2
# The origin is a saddle — minimum along x, maximum along y
def f(xy):  return xy[0]**2 - xy[1]**2
def grad(xy): return np.array([2*xy[0], -2*xy[1]])

# Hessian is [[2, 0], [0, -2]] — eigenvalues 2 and -2 → indefinite → saddle
H = np.array([[2.0, 0.0], [0.0, -2.0]])
print(f"H eigenvalues = {np.linalg.eigvalsh(H)} → mixed signs → saddle\n")

# Start very close to the saddle along the negative-curvature direction
xy = np.array([1e-3, 1e-3])
print("Gradient descent on f(x, y) = x^2 - y^2 — diverges along y:")
for step in range(8):
    xy = xy - 0.1 * grad(xy)
    print(f"  step {step}: xy = {xy}")

print("\nThe x-coordinate (positive curvature) shrinks toward 0.")
print("The y-coordinate (negative curvature) GROWS — gradient descent is escaping the saddle in that direction.")
print("In high dimensions, this 'escape' is most of what makes training work.")


## Where this appears in ML

Optimization is *the* thing that makes ML work. Every chapter of every modern textbook lives or dies by the algorithms in this notebook.

- **Every training loop.** Forward → loss → backprop → optimizer step. The optimizer is one of: SGD, SGD + momentum, Adam, AdamW, RMSProp, L-BFGS, …
- **Learning-rate scheduling.** Cosine decay, linear warmup, "one-cycle" — calculated reductions in `α` to navigate curvature regions. Theoretically motivated by the convergence rates above.
- **Adam vs SGD.** Convex / smooth settings favor SGD-with-momentum (it finds flat minima, which generalize better). Adam dominates in transformer training because it adapts to per-parameter scale, which matters for layer normalization interactions.
- **Sharpness-Aware Minimization (SAM).** Add a small perturbation step before computing the gradient, aiming to find *flat* minima with small Hessian eigenvalues. Directly targets the second-order geometry from this notebook.
- **Trust-region methods.** Newton-style methods that cap step size based on how well the quadratic Taylor approximation matched reality. Used in classical RL (TRPO).
- **Convex optimization solvers.** Linear regression, logistic regression, SVMs, lasso — all convex; all solved with deterministic algorithms (interior-point, coordinate descent, ADMM).
- **L-BFGS in classical ML.** Default for medium-scale logistic regression in `scikit-learn`, for many MLE problems, for fitting Gaussian processes.
- **Natural gradient and K-FAC.** Used in reinforcement learning and second-order deep learning experiments — closer to Newton, cheaper to apply.
- **Loss landscape research.** Mode connectivity, lottery-ticket hypothesis, neural tangent kernel — all involve studying the geometry of `L(w)` empirically. Convexity language is part of the vocabulary even where convexity itself doesn't apply.

Next notebook: **integration and expectation** — closing the calculus repo with the integration half of FTC, the change-of-variables formula, and expectation as an integral. This bridges into the probability and statistics repo.
